# 강아지 견종 분류 FastAPI 서버

Kaggle Notebook에서 바로 실행할 수 있는 견종 분류 API 예제입니다.

- 입력 이미지에서 `rembg`로 배경을 제거합니다.
- `Model/dog_breed_detect.safetensors` 모델로 견종을 분류합니다.
- 10개 학습 견종의 확률이 낮으면 `기타`로 반환합니다.
- FastAPI `/predict` 엔드포인트로 외부에서 이미지를 업로드하고 JSON 결과를 받을 수 있습니다.

> Kaggle에서 외부 호출을 받으려면 보통 `ngrok` 같은 터널이 필요합니다. 아래 셀은 `NGROK_AUTHTOKEN` 환경변수가 있으면 공개 URL을 자동으로 만듭니다.

In [ ]:
# Kaggle 환경에서 필요한 패키지를 설치합니다.
# Internet 옵션이 꺼져 있으면 이 셀은 실패할 수 있으니 Kaggle Notebook Settings에서 Internet을 켜주세요.
!pip -q install "transformers>=4.40" safetensors fastapi uvicorn python-multipart pillow rembg onnxruntime nest-asyncio pyngrok

In [ ]:
import io
import json
import os
import threading
import time
from pathlib import Path
from typing import Any, Dict, List

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from PIL import Image
from rembg import remove
from safetensors.torch import load_file
from transformers import ViTConfig, ViTForImageClassification, ViTImageProcessor

# Notebook 안에서 uvicorn 이벤트 루프를 안전하게 실행하기 위한 설정입니다.
nest_asyncio.apply()

In [ ]:
# Kaggle 입력 데이터셋 또는 로컬 프로젝트에서 모델 폴더를 자동으로 찾습니다.
# Kaggle에 업로드할 때는 Model/config.json, Model/preprocessor_config.json,
# Model/dog_breed_detect.safetensors가 포함된 데이터셋을 Add Input으로 연결하세요.

MODEL_FILE_NAME = "dog_breed_detect.safetensors"

def find_model_dir() -> Path:
    candidates = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for base in candidates:
        if not base.exists():
            continue
        for model_file in base.rglob(MODEL_FILE_NAME):
            model_dir = model_file.parent
            if (model_dir / "config.json").exists() and (model_dir / "preprocessor_config.json").exists():
                return model_dir
    raise FileNotFoundError(
        "Model/dog_breed_detect.safetensors, config.json, preprocessor_config.json를 찾지 못했습니다. "
        "Kaggle Add Input에 모델 파일이 들어있는 데이터셋을 연결했는지 확인하세요."
    )

MODEL_DIR = find_model_dir()
MODEL_PATH = MODEL_DIR / MODEL_FILE_NAME
print(f"모델 폴더: {MODEL_DIR}")
print(f"모델 파일: {MODEL_PATH}")

In [ ]:
# 모델 라벨을 서비스에서 보여줄 한글 견종명으로 변환합니다.
# 모델은 10개 견종만 학습되어 있으므로, 최고 확률이 임계값보다 낮으면 '기타'로 처리합니다.

LABEL_KO = {
    "maltese": "몰티즈",
    "poodle": "푸들",
    "mixed": "믹스견",
    "pomeranian": "포메라니안",
    "bichon_frise": "비숑 프리제",
    "chihuahua": "치와와",
    "shih_tzu": "시츄",
    "jindo": "진돗개",
    "yorkshire_terrier": "요크셔테리어",
    "golden_retriever": "골든 리트리버",
}

# 기타 판정 기준입니다. 실제 운영 전 검증 데이터로 0.5~0.8 범위에서 조정하는 것을 권장합니다.
OTHER_THRESHOLD = float(os.getenv("DOG_BREED_OTHER_THRESHOLD", "0.65"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")
print(f"기타 임계값: {OTHER_THRESHOLD}")

In [ ]:
# config.json과 preprocessor_config.json은 Hugging Face Transformers 형식입니다.
# safetensors 파일명이 model.safetensors가 아니므로 load_file로 직접 가중치를 로드합니다.

config = ViTConfig.from_pretrained(MODEL_DIR)
processor = ViTImageProcessor.from_pretrained(MODEL_DIR)
model = ViTForImageClassification(config)
state_dict = load_file(str(MODEL_PATH), device="cpu")
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)

if missing_keys:
    print("누락된 가중치 키:", missing_keys[:10])
if unexpected_keys:
    print("예상하지 못한 가중치 키:", unexpected_keys[:10])

model.to(device)
model.eval()

id2label = {int(k): v for k, v in config.id2label.items()}
print("모델 라벨:", id2label)

In [ ]:
def load_image_from_bytes(image_bytes: bytes) -> Image.Image:
    """업로드된 이미지 바이트를 PIL RGB 이미지로 변환합니다."""
    try:
        return Image.open(io.BytesIO(image_bytes)).convert("RGB")
    except Exception as exc:
        raise ValueError("이미지 파일을 읽을 수 없습니다.") from exc


def remove_background(image: Image.Image) -> Image.Image:
    """rembg로 배경을 제거한 뒤, 투명 영역을 흰색으로 합성합니다."""
    rgba_image = image.convert("RGBA")
    dog_only = remove(rgba_image)
    if not isinstance(dog_only, Image.Image):
        dog_only = Image.open(io.BytesIO(dog_only)).convert("RGBA")
    else:
        dog_only = dog_only.convert("RGBA")

    white_background = Image.new("RGBA", dog_only.size, (255, 255, 255, 255))
    composed = Image.alpha_composite(white_background, dog_only)
    return composed.convert("RGB")


def classify_dog_breed(image: Image.Image, top_k: int = 3) -> Dict[str, Any]:
    """배경 제거 이미지로 견종을 예측하고 API 응답 딕셔너리를 만듭니다."""
    inputs = processor(images=image, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0].detach().cpu()

    best_id = int(torch.argmax(probabilities).item())
    best_label_en = id2label[best_id]
    best_confidence = float(probabilities[best_id].item())

    predicted_breed = LABEL_KO.get(best_label_en, best_label_en)
    is_other = best_confidence < OTHER_THRESHOLD
    if is_other:
        predicted_breed = "기타"

    top_indices = torch.topk(probabilities, k=min(top_k, len(probabilities))).indices.tolist()
    top_predictions: List[Dict[str, Any]] = []
    for class_id in top_indices:
        label_en = id2label[int(class_id)]
        top_predictions.append({
            "label": label_en,
            "breed": LABEL_KO.get(label_en, label_en),
            "confidence": round(float(probabilities[class_id].item()), 6),
        })

    return {
        "breed": predicted_breed,
        "label": "other" if is_other else best_label_en,
        "confidence": round(best_confidence, 6),
        "is_other": is_other,
        "other_threshold": OTHER_THRESHOLD,
        "top_predictions": top_predictions,
    }


def predict_from_bytes(image_bytes: bytes) -> Dict[str, Any]:
    """이미지 로드 -> 배경 제거 -> 견종 분류 전체 파이프라인입니다."""
    original_image = load_image_from_bytes(image_bytes)
    dog_image = remove_background(original_image)
    result = classify_dog_breed(dog_image)
    result["background_removed"] = True
    result["image_size"] = {"width": original_image.width, "height": original_image.height}
    return result

In [ ]:
# FastAPI 앱을 정의합니다.
# 외부 서비스에서는 POST /predict 에 multipart/form-data 형식으로 image 파일을 보내면 됩니다.

app = FastAPI(title="Dog Breed Detection API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
def root() -> Dict[str, Any]:
    return {
        "message": "Dog Breed Detection API",
        "predict_endpoint": "POST /predict",
        "labels": list(LABEL_KO.values()) + ["기타"],
    }


@app.get("/health")
def health() -> Dict[str, Any]:
    return {"status": "ok", "device": str(device), "model_dir": str(MODEL_DIR)}


@app.get("/labels")
def labels() -> Dict[str, Any]:
    return {"labels": list(LABEL_KO.values()) + ["기타"], "threshold": OTHER_THRESHOLD}


@app.post("/predict")
async def predict(image: UploadFile = File(...)) -> JSONResponse:
    if not image.content_type or not image.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="이미지 파일만 업로드할 수 있습니다.")

    image_bytes = await image.read()
    if not image_bytes:
        raise HTTPException(status_code=400, detail="업로드된 파일이 비어 있습니다.")

    try:
        result = predict_from_bytes(image_bytes)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"예측 중 오류가 발생했습니다: {exc}") from exc

    result["filename"] = image.filename
    return JSONResponse(result)

In [ ]:
# 선택 사항: 서버 실행 전 간단한 로컬 이미지 테스트입니다.
# TEST_IMAGE_PATH에 Kaggle input 또는 working 경로의 강아지 이미지 파일을 넣고 실행하세요.

TEST_IMAGE_PATH = ""  # 예: "/kaggle/input/sample-dog/dog.jpg"

if TEST_IMAGE_PATH:
    with open(TEST_IMAGE_PATH, "rb") as f:
        test_result = predict_from_bytes(f.read())
    print(json.dumps(test_result, ensure_ascii=False, indent=2))
else:
    print("TEST_IMAGE_PATH가 비어 있어 테스트를 건너뜁니다.")

In [ ]:
# FastAPI 서버를 백그라운드 스레드로 실행합니다.
# Kaggle Notebook 내부에서는 http://127.0.0.1:8000/docs 로 Swagger 문서를 확인할 수 있습니다.

HOST = "0.0.0.0"
PORT = int(os.getenv("PORT", "8000"))

def run_api_server() -> None:
    uvicorn.run(app, host=HOST, port=PORT, log_level="info")

server_thread = threading.Thread(target=run_api_server, daemon=True)
server_thread.start()
time.sleep(2)

print(f"FastAPI 서버 실행 중: http://127.0.0.1:{PORT}")
print(f"Swagger 문서: http://127.0.0.1:{PORT}/docs")

In [ ]:
# 외부 호출용 공개 URL을 만들고 싶다면 Kaggle Secrets 또는 환경변수에 NGROK_AUTHTOKEN을 등록하세요.
# 토큰이 없으면 이 셀은 로컬 서버 URL만 출력합니다.

public_url = None
ngrok_token = os.getenv("NGROK_AUTHTOKEN")

if ngrok_token:
    from pyngrok import ngrok

    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(PORT, bind_tls=True)
    public_url = tunnel.public_url
    print(f"공개 API URL: {public_url}")
    print(f"예측 엔드포인트: {public_url}/predict")
else:
    print("NGROK_AUTHTOKEN이 없어 공개 URL을 만들지 않았습니다.")
    print(f"Kaggle 내부 URL: http://127.0.0.1:{PORT}/predict")

In [ ]:
# 외부에서 호출하는 예시입니다.
# 아래 명령에서 YOUR_PUBLIC_URL을 위 셀의 공개 URL로 바꾸고, dog.jpg 경로를 실제 파일 경로로 바꾸세요.

print("curl 예시:")
print('curl -X POST "YOUR_PUBLIC_URL/predict" -F "image=@dog.jpg"')

print("\nPython requests 예시:")
print('''
import requests

url = "YOUR_PUBLIC_URL/predict"
with open("dog.jpg", "rb") as f:
    response = requests.post(url, files={"image": f})

print(response.json())
''')